In [5]:
import csv
import glob
import os
import re
from bs4 import BeautifulSoup

def process_all_html_files(directory_path):
    # Find all .html files in the specified directory
    html_files = glob.glob(os.path.join(directory_path, '*.html'))

    if not html_files:
        print("No HTML files found in the directory.")
        return

    for html_file in html_files:
        print(f"Processing: {os.path.basename(html_file)}")

        # Determine the output filename (same base name, but with .csv extension)
        base_name = os.path.splitext(html_file)[0]
        output_csv_path = base_name + '.csv'

        # Read the current HTML file
        with open(html_file, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        song_items = soup.find_all('li', class_='search_result')

        extracted_data = []

        # Extract data, starting the index at 1 for every new file
        for index, item in enumerate(song_items, start=1):
            # 1. Extract URL
            a_tag = item.find('a', class_='song_link')
            if not a_tag:
                continue
            url = a_tag.get('href', '').strip()

            # 2. Extract Title
            title_tag = item.find('span', class_='song_title')
            title = title_tag.text.strip() if title_tag else ''
            title = title.replace('\xa0', ' ') # Clean up non-breaking spaces

            # 3. Extract Artist
            artist_tag = item.find('span', class_='artist_name')
            artist = artist_tag.text.strip() if artist_tag else ''
            artist = artist.replace('\xa0', ' ')
            artist = re.sub(r'\s+', ' ', artist) # Clean up extra white spaces

            # 4. Generate custom slug (e.g., 001-Artist-song-title)
            url_slug = url.split('/')[-1].replace('-lyrics', '')
            formatted_id = f"{index:03d}"
            custom_slug = f"{formatted_id}-{url_slug}"

            extracted_data.append([index, title, artist, url, custom_slug])

        # Write to the specific CSV file
        if extracted_data:
            with open(output_csv_path, 'w', encoding='utf-8', newline='') as f:
                # Using '\t' (tab) as the delimiter based on your original example
                writer = csv.writer(f, delimiter='\t')
                writer.writerows(extracted_data)
            print(f"  -> Saved {len(extracted_data)} songs to {os.path.basename(output_csv_path)}")
        else:
            print(f"  -> No songs found. Skipped creating {os.path.basename(output_csv_path)}")

# --- Instructions to run ---
if __name__ == "__main__":
    # '.' means it will look in the current folder where the script is running.
    input_directory = '.'

    process_all_html_files(input_directory)

Processing: Everything in Rap _ Genius.html
  -> Saved 999 songs to Everything in Rap _ Genius.csv
Processing: Everything in French Rap _ Genius.html
  -> Saved 999 songs to Everything in French Rap _ Genius.csv
